## One-vs-Rest Binary Classifier Approach for Benthic Habitat Mapping

Instead of training a single multiclass model to predict all 5 habitat classes at once, this notebook trains one dedicated binary classifier per class. Each binary classifier answers a single focused question,"is this point SGAM or not?", "is this ALG or not?" allowing each model to fully specialise on the boundary between one class and everything else, rather than trying to separate all 5 classes simultaneously.
For each class, two binary classifiers are trained in parallel across 5-fold stratified CV LightGBM and XGBoost whose predicted probabilities are blended 50/50. This reduces correlated errors since the two models use fundamentally different tree-building strategies (leaf-wise vs level-wise).

Each binary classifier gets its own independently tuned decision threshold, found by sweeping the precision-recall curve on OOF predictions and picking the threshold that maximises binary F1 for that class. This means a class like FMAT which is rare and hard to detect can have a lower threshold to catch more positives, while a dominant class like NVB can have a stricter one.

Final predictions are resolved as follows: if exactly one classifier fires positive, that's the label directly. If multiple fire, the one with the highest probability among them wins. If none fire, the class with the highest raw probability is used as a fallback. The OOF breakdown of clean vs conflict vs abstention cases is printed after threshold tuning, so you can judge how well the thresholds are working ideally 85%+ of points should be clean single-fire predictions.

## Imports

In [190]:
import rasterio
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, precision_recall_curve
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from xgboost import XGBClassifier

## Importing data

In [191]:
back = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/backscatter.tif")
bath = rasterio.open("/kaggle/input/competitions/geohab-mlwg-competition-2026/MBES/bathymetry.tif")
train_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/geohab-mlwg-competition-2026/test.csv')

In [192]:
print('Train set: ', train_df.shape)
display(train_df.head())

print('Test set: ', test_df.shape)
display(test_df.head())

back_data = back.read(1)
print('Backscatter data: ',back_data.shape)
print(back_data[:5, :5])

bath_data = bath.read(1)
print('Bathymestry data: ', bath_data.shape)
print(bath_data[:5, :5])

Train set:  (6256, 3)


,class,x,y
0,NVB,453594.477237,5.679192e+06
1,FMAT,453561.906453,5.679109e+06
2,ALG,453744.452238,5.679033e+06
3,ALG,453863.445302,5.679038e+06
4,ALG,453964.611906,5.679017e+06


Test set:  (98, 3)


,ID,x,y
0,1,453702.166779,5.679044e+06
1,2,454126.252800,5.678999e+06
2,3,453957.881092,5.678942e+06
3,4,453798.917484,5.678955e+06
4,5,453520.953671,5.679124e+06


Backscatter data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]
Bathymestry data:  (4040, 4743)
[[-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]
 [-10000. -10000. -10000. -10000. -10000.]]


## Extracting Features:

### Grid based features

In [193]:
def get_neighborhood_features(x, y):
    row, col = bath.index(x, y)
    
    features = []
    
    for dr in [-12, 0, 12]:
        for dc in [-12, 0, 12]:
            r = row + dr
            c = col + dc
            
            try:
                depth = bath_data[r, c]
                scatter = back_data[r, c]
            except:
                depth = 0
                scatter = 0
            
            features.append(depth)
            features.append(scatter)
    
    return features

In [194]:
features = train_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

train_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

train_df[train_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [195]:
print(train_feature_cols)
train_df.head()

['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17']


,class,x,y,f0,f1,f2,f3,f4,f5,f6,...,f8,f9,f10,f11,f12,f13,f14,f15,f16,f17
0,NVB,453594.477237,5.679192e+06,-10000.000000,-10000.000000,-10000.000000,-10000.000000,-10000.000000,-10000.000000,-2.667833,...,-2.822883,-18.669088,-3.003832,-16.779381,-3.052902,-17.719353,-3.153770,-18.979156,-3.217835,-21.499985
1,FMAT,453561.906453,5.679109e+06,-5.861867,-24.649494,-5.705794,-19.609058,-5.636959,-19.928892,-5.808707,...,-5.668992,-20.558794,-5.697957,-22.129887,-6.120852,-20.558794,-5.797802,-21.189917,-6.232965,-21.189917
2,ALG,453744.452238,5.679033e+06,-9.233783,-24.339426,-8.520893,-24.019592,-8.426841,-24.339426,-9.579664,...,-9.288988,-23.389690,-9.054879,-23.709524,-9.672694,-27.169104,-10.006989,-21.189917,-10.029820,-23.709524
3,ALG,453863.445302,5.679038e+06,-6.856915,-24.339426,-6.731852,-24.339426,-6.698797,-25.599232,-7.546973,...,-7.263793,-21.189917,-6.937677,-22.449720,-8.047904,-19.609058,-7.883994,-18.039186,-7.642729,-21.819818
4,ALG,453964.611906,5.679017e+06,-8.769996,-24.339426,-7.727922,-20.238960,-7.976172,-24.649494,-9.685985,...,-9.784807,-22.449720,-8.631302,-28.749962,-10.117740,-25.599232,-10.628894,-27.169104,-10.871862,-27.488937


### More spatial features

In [196]:
def get_advanced_features(x, y):
    row, col = bath.index(x, y)
    
    depths = []
    scatters = []
    
    for dr in [-10, 0, 10]:
        for dc in [-10, 0, 10]:
            r = row + dr
            c = col + dc
            
            if 0 <= r < bath_data.shape[0] and 0 <= c < bath_data.shape[1]:
                depths.append(bath_data[r, c])
                scatters.append(back_data[r, c])
            else:
                depths.append(0)
                scatters.append(0)
    
    return [
        np.mean(depths),
        np.std(depths),
        np.min(depths),
        np.max(depths),
        np.mean(scatters),
        np.std(scatters),
        np.min(scatters),
        np.max(scatters),
    ]

In [197]:
features = train_df.apply(
    lambda r: get_advanced_features(r['x'], r['y']),
    axis=1
)

adv_feature_cols = [
    'depth_mean', 'depth_std', 'depth_min', 'depth_max',
    'scatter_mean', 'scatter_std', 'scatter_min', 'scatter_max'
]

train_df[adv_feature_cols] = pd.DataFrame(features.tolist(), index=train_df.index)

In [198]:
print(adv_feature_cols)

['depth_mean', 'depth_std', 'depth_min', 'depth_max', 'scatter_mean', 'scatter_std', 'scatter_min', 'scatter_max']


In [199]:
target = 'class'

X = train_df[train_feature_cols + adv_feature_cols]
y = train_df[target].map({'SGAM': 0, 'NVB': 1, 'SGZ': 2, 'ALG': 3, 'FMAT': 4})

In [200]:
features = test_df.apply(
    lambda r: get_neighborhood_features(r['x'], r['y']),
    axis=1
)

test_feature_cols = [f'f{i}' for i in range(len(features.iloc[0]))]

test_df[test_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

In [201]:
features = test_df.apply(
    lambda r: get_advanced_features(r['x'], r['y']),
    axis=1
)

test_df[adv_feature_cols] = pd.DataFrame(features.tolist(), index=test_df.index)

In [202]:
X_test = test_df[test_feature_cols + adv_feature_cols]

## One-vs-Rest Binary Classifiers with Threshold Tuning

In [203]:
CLASSES = ['SGAM', 'NVB', 'SGZ', 'ALG', 'FMAT']
SEED    = 42
N_FOLDS = 5

# Create the mapping
class_to_idx = {c: i for i, c in enumerate(CLASSES)}

# FIX: Map from the original text column in train_df, not the already-mapped 'y'
y_enc = train_df['class'].map(class_to_idx).values 

# Optional: verify there are no NaNs now
print(f"Any NaNs in y_enc? {np.isnan(y_enc).any()}")

Any NaNs in y_enc? False


In [204]:
# ── Shared hyperparameters ────────────────────────────────────────────────────
lgb_params = {
   'n_estimators': 1879,
 	'learning_rate': 0.0765079139135192,
 	'num_leaves': 57,
 	'max_depth': 13,
 	'min_child_samples': 23,
 	'subsample': 0.9491400225657889,
 	'colsample_bytree': 0.6671968380388436,
	'reg_alpha': 0.002067039466798589,
 	'reg_lambda': 0.004458960681919556,
    "random_state": SEED,
    "device": "gpu",
    "n_jobs": -1,
    "verbose": -1,
}

xgb_params = {
    "n_estimators": 2000,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.80,
    "colsample_bytree": 0.70,
    "min_child_weight": 3,
    "reg_alpha": 0.5,
    "reg_lambda": 2.0,
    "gamma": 0.05,
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "early_stopping_rounds": 50,
    "tree_method": "hist",
    "device": "cuda",
    "verbosity": 0,
    "random_state": SEED,
    "n_jobs": -1,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

# Storage: OOF probs and test probs per class
oof_probs  = np.zeros((len(X), len(CLASSES)))   # OOF positive probs per class
test_probs = np.zeros((len(X_test), len(CLASSES)))  # test positive probs per class

print('Training one LGB + one XGB binary classifier per class...\n')

for cls_idx, cls_name in enumerate(CLASSES):
    # Binary target: 1 if this point belongs to cls_name, else 0
    y_bin = (y_enc == cls_idx).astype(int)
    pos_ratio = y_bin.mean()
    print(f'Class {cls_name} ({pos_ratio:.1%} positive)')

    oof_cls  = np.zeros(len(X))
    test_cls = np.zeros(len(X_test))

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y_enc)):
        X_tr, y_tr_bin = X.iloc[train_idx], y_bin[train_idx]
        X_va, y_va_bin = X.iloc[val_idx],   y_bin[val_idx]
        sw_tr = compute_sample_weight('balanced', y=y_tr_bin)

        # ── LightGBM binary ───────────────────────────────────────────────────
        lgb_model = LGBMClassifier(
            **lgb_params,
            objective='binary',
            # class_weight='balanced',
        )
        lgb_model.fit(
            X_tr, y_tr_bin,
            eval_set=[(X_va, y_va_bin)],
            # callbacks=[early_stopping(50, verbose=False), log_evaluation(0)]
        )
        lgb_oof_prob  = lgb_model.predict_proba(X_va)[:, 1]
        lgb_test_prob = lgb_model.predict_proba(X_test)[:, 1]

        # ── XGBoost binary ────────────────────────────────────────────────────
        xgb_model = XGBClassifier(**xgb_params)
        xgb_model.fit(
            X_tr, y_tr_bin,
            eval_set=[(X_va, y_va_bin)],
            sample_weight=sw_tr,
            verbose=False
        )
        xgb_oof_prob  = xgb_model.predict_proba(X_va)[:, 1]
        xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]

        # Blend LGB + XGB probs 50/50
        oof_cls[val_idx] = 0.5 * lgb_oof_prob + 0.5 * xgb_oof_prob
        test_cls        += (0.5 * lgb_test_prob + 0.5 * xgb_test_prob) / N_FOLDS

    oof_probs[:, cls_idx]  = oof_cls
    test_probs[:, cls_idx] = test_cls

    # OOF binary F1 at default 0.5 threshold (pre-tuning baseline)
    default_f1 = f1_score(y_bin, (oof_cls >= 0.5).astype(int))
    print(f'  OOF binary F1 @ 0.50 threshold: {default_f1:.4f}\n')

print('Done training all binary classifiers.')

Training one LGB + one XGB binary classifier per class...

Class SGAM (2.7% positive)
  OOF binary F1 @ 0.50 threshold: 0.9141

Class NVB (48.5% positive)
  OOF binary F1 @ 0.50 threshold: 0.9633

Class SGZ (13.2% positive)
  OOF binary F1 @ 0.50 threshold: 0.9394

Class ALG (10.9% positive)
  OOF binary F1 @ 0.50 threshold: 0.9279

Class FMAT (24.7% positive)
  OOF binary F1 @ 0.50 threshold: 0.9482

Done training all binary classifiers.


In [205]:
# ── Step 1: Tune decision threshold per class on OOF ─────────────────────────
# For each binary classifier, find the threshold that maximises binary F1 on OOF.
# This is done independently per class — each gets its own best threshold.

best_thresholds = {}

print('Tuning thresholds on OOF probabilities...\n')
for cls_idx, cls_name in enumerate(CLASSES):
    y_bin    = (y_enc == cls_idx).astype(int)
    oof_cls  = oof_probs[:, cls_idx]

    # Sweep all candidate thresholds from precision-recall curve
    precisions, recalls, thresholds = precision_recall_curve(y_bin, oof_cls)
    # F1 at each threshold
    f1_scores = 2 * precisions * recalls / (precisions + recalls + 1e-9)
    best_idx  = np.argmax(f1_scores[:-1])  # last element has no matching threshold
    best_thr  = thresholds[best_idx]
    best_f1   = f1_scores[best_idx]

    best_thresholds[cls_name] = best_thr
    print(f'  {cls_name}: best threshold = {best_thr:.4f}  |  binary F1 = {best_f1:.4f}')

# ── Step 2: Apply tuned thresholds to OOF to measure multiclass F1 ────────────
def resolve_predictions(probs_matrix, thresholds, classes):
    """
    probs_matrix : (N, n_classes) array of positive probabilities
    thresholds   : dict {class_name: threshold}
    Returns      : array of predicted class names
    """
    n = probs_matrix.shape[0]
    thr_arr = np.array([thresholds[c] for c in classes])   # shape (n_classes,)
    fires   = probs_matrix >= thr_arr                       # bool (N, n_classes)
    n_fires = fires.sum(axis=1)                             # how many classifiers fire

    preds = np.empty(n, dtype=object)

    for i in range(n):
        fired_classes = np.where(fires[i])[0]

        if len(fired_classes) == 1:
            # Clean: exactly one classifier fires
            preds[i] = classes[fired_classes[0]]

        elif len(fired_classes) > 1:
            # Conflict: multiple fire → pick highest prob among those
            best = fired_classes[np.argmax(probs_matrix[i, fired_classes])]
            preds[i] = classes[best]

        else:
            # Abstention: none fire → argmax across all classes
            preds[i] = classes[np.argmax(probs_matrix[i])]

    return preds

# OOF multiclass F1
oof_preds    = resolve_predictions(oof_probs, best_thresholds, CLASSES)
oof_true     = np.array([CLASSES[i] for i in y_enc])
oof_f1       = f1_score(oof_true, oof_preds, average='weighted')

# Breakdown: how many clean / conflict / abstentions
thr_arr  = np.array([best_thresholds[c] for c in CLASSES])
fires    = oof_probs >= thr_arr
n_fires  = fires.sum(axis=1)
n_clean      = (n_fires == 1).sum()
n_conflict   = (n_fires  > 1).sum()
n_abstention = (n_fires == 0).sum()

print(f'\nOOF Weighted F1: {oof_f1:.4f}')
print(f'Clean (1 fires):     {n_clean} ({n_clean/len(X):.1%})')
print(f'Conflict (>1 fire):  {n_conflict} ({n_conflict/len(X):.1%})')
print(f'Abstention (0 fire): {n_abstention} ({n_abstention/len(X):.1%})')

# ── Step 3: Generate test predictions ─────────────────────────────────────────
final_preds = resolve_predictions(test_probs, best_thresholds, CLASSES)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    target: final_preds
})

submission.to_csv('submission_binary_ovr.csv', index=False)
print(f'\nSaved: submission_binary_ovr.csv')
print(f'Prediction distribution:\n{pd.Series(final_preds).value_counts()}')

Tuning thresholds on OOF probabilities...

  SGAM: best threshold = 0.4649  |  binary F1 = 0.9152
  NVB: best threshold = 0.4495  |  binary F1 = 0.9639
  SGZ: best threshold = 0.3911  |  binary F1 = 0.9403
  ALG: best threshold = 0.5421  |  binary F1 = 0.9281
  FMAT: best threshold = 0.4746  |  binary F1 = 0.9497

OOF Weighted F1: 0.9512
Clean (1 fires):     5995 (95.8%)
Conflict (>1 fire):  137 (2.2%)
Abstention (0 fire): 124 (2.0%)

Saved: submission_binary_ovr.csv
Prediction distribution:
NVB     43
ALG     20
FMAT    19
SGAM     9
SGZ      7
Name: count, dtype: int64
